# Feature engineering for property assessment valuation

This notebook prepares features for predicting log-transformed Calgary
property assessment values. Steps include:

1. Loading and cleaning assessment data
2. Log-transforming the target variable
3. Encoding categorical features (property class, community, land use)
4. Computing community-level aggregations
5. Land use frequency encoding
6. Feature correlation analysis

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.insert(0, '..')
from src.data_loader import load_or_fetch_data, preprocess_data, engineer_features
from src.model import CATEGORICAL_FEATURES, NUMERICAL_FEATURES, TARGET, prepare_model_data

DATA_DIR = str(Path('..') / 'data')
pd.set_option('display.max_columns', 40)
print('Setup complete.')

## 1. Load assessment data

In [ ]:
raw = load_or_fetch_data(DATA_DIR, limit=100000)
print(f'Raw records: {len(raw):,}')
print(f'Columns: {list(raw.columns)}')
raw.head()

In [ ]:
df = preprocess_data(raw)
print(f'After cleaning: {len(df):,} records')
print(f'assessed_value range: ${df["assessed_value"].min():,.0f} -- ${df["assessed_value"].max():,.0f}')
df.describe()

## 2. Log-transform the target variable

Property values are right-skewed. Applying `log1p` makes the distribution
closer to normal, which improves regression model performance.

In [ ]:
df = engineer_features(df)

fig = make_subplots(rows=1, cols=2, subplot_titles=['assessed_value (raw)', 'log_value (log-transformed)'])
fig.add_trace(go.Histogram(x=df['assessed_value'], nbinsx=50, marker_color='steelblue', showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=df['log_value'], nbinsx=50, marker_color='darkorange', showlegend=False), row=1, col=2)
fig.update_layout(title='Target variable distribution: raw vs. log-transformed', height=350)
fig.show()

print(f'Skewness (raw):  {df["assessed_value"].skew():.2f}')
print(f'Skewness (log):  {df["log_value"].skew():.2f}')

## 3. Encode categorical features

The model uses three categorical features: `property_class`, `community`,
and `land_use_designation`. We inspect their distributions before encoding.

In [ ]:
if 'property_class' in df.columns:
    pc = df['property_class'].value_counts().reset_index()
    pc.columns = ['Property class', 'Count']
    fig = px.bar(pc, x='Count', y='Property class', orientation='h',
                 title='Property class distribution',
                 color_discrete_sequence=['teal'])
    fig.update_layout(height=300)
    fig.show()

In [ ]:
if 'community' in df.columns:
    top_communities = df['community'].value_counts().head(20).reset_index()
    top_communities.columns = ['Community', 'Count']
    fig = px.bar(top_communities, x='Count', y='Community', orientation='h',
                 title='Top 20 communities by property count',
                 color_discrete_sequence=['steelblue'])
    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=500)
    fig.show()

print(f'Unique communities: {df["community"].nunique()}')

In [ ]:
if 'land_use_designation' in df.columns:
    lu = df['land_use_designation'].value_counts().head(15).reset_index()
    lu.columns = ['Land use', 'Count']
    fig = px.bar(lu, x='Count', y='Land use', orientation='h',
                 title='Top 15 land use designations',
                 color_discrete_sequence=['mediumseagreen'])
    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=400)
    fig.show()

## 4. Community-level aggregations

The `engineer_features` function creates three community-level features:

- `community_avg_value`: mean assessed value in the community
- `community_median_value`: median assessed value
- `community_property_count`: number of properties

In [ ]:
agg_cols = ['community_avg_value', 'community_median_value', 'community_property_count']
available_agg = [c for c in agg_cols if c in df.columns]

if available_agg:
    community_agg = df.drop_duplicates(subset='community')[['community'] + available_agg]
    community_agg = community_agg.sort_values('community_avg_value', ascending=False)
    print(f'Communities with aggregated features: {len(community_agg)}')
    community_agg.head(15)

In [ ]:
if 'community_avg_value' in df.columns:
    fig = px.scatter(
        community_agg.head(40), x='community_property_count', y='community_avg_value',
        hover_name='community', size='community_median_value',
        title='Community average value vs. property count (top 40)',
        labels={'community_property_count': 'Property count', 'community_avg_value': 'Avg assessed value ($)'},
        color='community_avg_value', color_continuous_scale='YlOrRd',
    )
    fig.update_layout(height=450)
    fig.show()

## 5. Land use frequency

Each land use designation is mapped to its city-wide count, acting as a
proxy for how common that zoning type is.

In [ ]:
if 'land_use_frequency' in df.columns:
    print(f'land_use_frequency range: {df["land_use_frequency"].min()} -- {df["land_use_frequency"].max()}')
    fig = px.histogram(df, x='land_use_frequency', nbins=30,
                       title='Land use frequency distribution',
                       labels={'land_use_frequency': 'Frequency'},
                       color_discrete_sequence=['darkorange'])
    fig.update_layout(height=350)
    fig.show()

## 6. Feature correlations

Check how the engineered features correlate with the log-transformed target.

In [ ]:
# Prepare model data to get encoded features
X, y, label_encoders, feature_names = prepare_model_data(df)

model_df = X.copy()
model_df['log_value'] = y.values

corr = model_df.corr()

fig = px.imshow(
    corr, text_auto='.2f', title='Feature correlation matrix',
    color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
)
fig.update_layout(height=550, width=650)
fig.show()

In [ ]:
# Correlation with target
target_corr = corr['log_value'].drop('log_value').sort_values(ascending=False)

fig = px.bar(
    x=target_corr.values, y=target_corr.index, orientation='h',
    title='Correlation with log_value',
    labels={'x': 'Correlation', 'y': 'Feature'},
    color=target_corr.values, color_continuous_scale='RdBu_r',
)
fig.update_layout(height=400)
fig.show()

target_corr

## 7. Final feature matrix

In [ ]:
print(f'Feature matrix: {X.shape}')
print(f'Features: {feature_names}')
print(f'Target (log_value) stats:')
print(f'  Mean: {y.mean():.4f}')
print(f'  Std:  {y.std():.4f}')
print(f'  Min:  {y.min():.4f}')
print(f'  Max:  {y.max():.4f}')
X.describe().T

## Summary

Feature engineering steps completed:

1. Loaded and cleaned property assessment data, removing outliers below the 1st and above the 99th percentile.
2. Applied `log1p` to `assessed_value`, reducing skewness significantly.
3. Inspected distributions of `property_class`, `community`, and `land_use_designation`.
4. Built community-level aggregates: average value, median value, and property count.
5. Computed `land_use_frequency` as a city-wide zoning prevalence feature.
6. Checked feature correlations -- `community_avg_value` and `community_median_value` correlate strongly with the target.

The feature matrix is ready for regression modelling in `03_modeling.ipynb`.